# Zone Hourly Demand Profile

**Grain:** one pickup zone × one pickup hour. A row exists when at least one valid trip starts there. `trip_count` measures demand; distance and duration describe those trips. The zone ID is the stable key, while names are descriptive. Unknown zone names are retained. This product can be queried by year/month partitions.


In [1]:
from pathlib import Path
import sys

# Find the project even when Jupyter starts in its notebooks directory.
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root / "notebooks" / "data_products"))
from product_utils import get_spark, register_integrated, publish_product

spark = get_spark()
register_integrated(spark)


D:\Master\data intensive\data-intensive-computing\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## Build the product

The SQL below defines the grain and metrics. Run this cell again to refresh the Delta table.


In [2]:
# Spark SQL keeps the grouping logic visible and reusable.
query = """
SELECT
    pickup_year,
    pickup_month,
    pickup_date,
    pickup_hour,
    pickup_location_id,
    COALESCE(pickup_zone, 'Unknown') AS pickup_zone,
    COALESCE(pickup_borough, 'Unknown') AS pickup_borough,
    COUNT(trip_id) AS trip_count,
    ROUND(AVG(trip_distance), 3) AS avg_trip_distance,
    ROUND(AVG(trip_duration_minutes), 3) AS avg_trip_duration_minutes
FROM integrated_taxi_trips
WHERE trip_id IS NOT NULL
GROUP BY pickup_year, pickup_month, pickup_date, pickup_hour,
         pickup_location_id, COALESCE(pickup_zone, 'Unknown'),
         COALESCE(pickup_borough, 'Unknown')
"""
result = spark.sql(query)
product = publish_product(spark, result, "zone_hourly_demand_profile", ('pickup_year', 'pickup_month'))
product.printSchema()
product.show(20, truncate=False)


Saved zone_hourly_demand_profile to D:\Master\data intensive\data-intensive-computing\data\processed\analytical_products\zone_hourly_demand_profile
root
 |-- pickup_year: integer (nullable = true)
 |-- pickup_month: integer (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: timestamp (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- trip_count: long (nullable = true)
 |-- avg_trip_distance: double (nullable = true)
 |-- avg_trip_duration_minutes: double (nullable = true)
 |-- source_table: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- refreshed_at: timestamp (nullable = true)
 |-- schema_version: string (nullable = true)

+-----------+------------+-----------+-------------------+------------------+-------------------------+--------------+----------+-----------------+-------------------------+---------------------+-------